# Keyword Research using Python

**Machine Learning · Data Analytics · Visualization**

Sources used:
- [PaulHex6/google-trends](https://github.com/PaulHex6/google-trends)
- [MuhammadAhmed-0/Keyword-Research-tool-python](https://github.com/MuhammadAhmed-0/Keyword-Research-tool-python)
- [SEO Sample Data (Kaggle)](https://www.kaggle.com/datasets/muhammetvarl/seo-sample-data)
- [Google Trends Dataset (Kaggle)](https://www.kaggle.com/datasets/dhruvildave/google-trends-dataset)

Run each cell top to bottom in **Google Colab** or **VS Code Jupyter**.

## 1. Install dependencies

In [ ]:
!pip install -q pandas numpy matplotlib seaborn plotly scikit-learn wordcloud pytrends

## 2. Imports and global setup

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 5)
os.makedirs('outputs', exist_ok=True)

## 3. Load SEO sample data
Download `seo_sample.csv` from the [Kaggle dataset](https://www.kaggle.com/datasets/muhammetvarl/seo-sample-data) and upload it. The notebook falls back to a built-in sample when the file is missing so it always runs.

In [ ]:
CSV_PATH = 'seo_sample.csv'
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    sample_keywords = [
        'python tutorial','machine learning','data science','deep learning',
        'pandas dataframe','numpy array','scikit learn','tensorflow',
        'keras model','data visualization','matplotlib chart','seaborn heatmap',
        'kaggle competition','jupyter notebook','google colab','linear regression',
        'logistic regression','kmeans clustering','tfidf vectorizer',
        'natural language processing','computer vision','neural network',
        'decision tree','random forest','xgboost','feature engineering',
        'data cleaning','exploratory data analysis','seo keywords',
        'keyword research tool','google trends','search volume','long tail keyword',
        'low competition keyword','content marketing','blog post ideas',
        'youtube seo','instagram seo','ecommerce seo','local seo',
        'backlink analysis','domain authority','page speed','mobile first',
        'schema markup','meta description','title tag optimization',
        'google analytics','search console','ahrefs alternative','semrush alternative'
    ]
    rng = np.random.default_rng(42)
    df = pd.DataFrame({
        'Keyword': sample_keywords,
        'Search Volume': rng.integers(150, 50000, size=len(sample_keywords)),
        'CPC': np.round(rng.uniform(15, 1000, size=len(sample_keywords)), 0),  # CPC in INR
        'Competition': np.round(rng.uniform(0.05, 0.95, size=len(sample_keywords)), 2),
        'Category': rng.choice(['Programming','ML','Data','SEO','Marketing'], size=len(sample_keywords))
    })
df.head()

## 4. Pull live Google Trends data with pytrends (optional)

In [ ]:
try:
    from pytrends.request import TrendReq
    pytrends = TrendReq(hl='en-US', tz=360)
    seeds = ['python','machine learning','data science','deep learning','seo']
    pytrends.build_payload(seeds, cat=0, timeframe='today 12-m')
    trends_df = pytrends.interest_over_time()
    if 'isPartial' in trends_df.columns:
        trends_df = trends_df.drop(columns=['isPartial'])
    trends_df.tail()
except Exception as e:
    print('pytrends skipped:', e)
    trends_df = None

## 5. Fallback: synthetic trends data when offline

In [ ]:
if trends_df is None:
    dates = pd.date_range('2025-01-01', periods=52, freq='W')
    trends_df = pd.DataFrame({
        'date': dates,
        'python': np.clip(70 + 10*np.sin(np.arange(52)/4) + np.random.randn(52)*4, 40, 100).astype(int),
        'machine learning': np.clip(60 + 8*np.cos(np.arange(52)/5) + np.random.randn(52)*3, 30, 100).astype(int),
        'data science': np.clip(55 + 5*np.sin(np.arange(52)/6) + np.random.randn(52)*3, 30, 100).astype(int),
        'deep learning': np.clip(45 + 12*np.sin(np.arange(52)/3) + np.random.randn(52)*4, 20, 100).astype(int),
        'seo': np.clip(35 + 6*np.cos(np.arange(52)/7) + np.random.randn(52)*3, 15, 90).astype(int),
    }).set_index('date')
trends_df.head()

## 6. EDA — quick look at the dataset

In [ ]:
print(df.shape)
print(df.isnull().sum())
df.describe()

## 7. Top 15 keywords by search volume

In [ ]:
top15 = df.nlargest(15, 'Search Volume')
plt.figure(figsize=(11,6))
sns.barplot(data=top15, x='Search Volume', y='Keyword', palette='viridis')
plt.title('Top 15 Keywords by Search Volume', fontsize=14, weight='bold')
plt.tight_layout(); plt.savefig('outputs/01_top_keywords.png', dpi=140); plt.show()

## 8. Google Trends — interest over time

In [ ]:
plt.figure(figsize=(11,5))
for col in trends_df.columns:
    plt.plot(trends_df.index, trends_df[col], label=col, linewidth=2)
plt.title('Google Trends — Interest Over Time', fontsize=14, weight='bold')
plt.xlabel('Date'); plt.ylabel('Interest (0-100)'); plt.legend(loc='upper right')
plt.tight_layout(); plt.savefig('outputs/02_trends_over_time.png', dpi=140); plt.show()

## 9. Correlation heatmap

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(df[['Search Volume','CPC','Competition']].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation between Search Volume, CPC and Competition')
plt.tight_layout(); plt.savefig('outputs/03_correlation_heatmap.png', dpi=140); plt.show()

## 10. CPC vs Search Volume scatter (bubble = competition)

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df['CPC'], df['Search Volume'], s=df['Competition']*400, c=df['Competition'], cmap='viridis', alpha=0.75, edgecolors='white')
plt.colorbar(label='Competition'); plt.xlabel('CPC (INR ₹)'); plt.ylabel('Search Volume')
plt.title('CPC vs Search Volume (bubble size = Competition)')
plt.tight_layout(); plt.savefig('outputs/04_cpc_vs_volume.png', dpi=140); plt.show()

## 11. Word cloud of all keywords

In [ ]:
wc = WordCloud(width=1000, height=500, background_color='white', colormap='viridis', collocations=False).generate(' '.join(df['Keyword']))
plt.figure(figsize=(12,6)); plt.imshow(wc, interpolation='bilinear'); plt.axis('off')
plt.title('Keyword Cloud', fontsize=14, weight='bold')
plt.tight_layout(); plt.savefig('outputs/06_wordcloud.png', dpi=140); plt.show()

## 12. Machine Learning — KMeans clustering on TF-IDF features

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['Keyword'])
km = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = km.fit_predict(X)
for k in range(5):
    print(f'Cluster {k}:', df[df.Cluster==k]['Keyword'].head(5).tolist())

## 13. Machine Learning — Linear Regression to predict Search Volume

In [ ]:
X = df[['CPC','Competition']]; y = df['Search Volume']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
print('R²  :', round(r2_score(y_test, y_pred), 3))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, y_pred)), 1))

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(y_test, y_pred, alpha=0.7, color='#7e22ce')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Search Volume'); plt.ylabel('Predicted Search Volume')
plt.title('Linear Regression — Actual vs Predicted')
plt.legend(); plt.tight_layout(); plt.savefig('outputs/08_regression.png', dpi=140); plt.show()

## 14. Save the enriched dataset

In [ ]:
df.to_csv('outputs/keywords_enriched.csv', index=False)
print('All charts + enriched CSV are in ./outputs/')